# Stage 2/3 — Colab T4 Execution & Resume Layer

This notebook provides a 100% self-contained, auto-resuming execution pipeline for Google Colab.

### Key Features:
1. **Zero Manual File Uploads**: Clones and updates the repository directly from GitHub (`https://github.com/aliakarma/Safe-Lie.git`).
2. **Google Drive Integration**: All checkpoints (`checkpoint.pt`) and JSONL logs (`rounds.jsonl`, `oracle.jsonl`) are written directly to Google Drive.
3. **Automatic Resumption**: If a Colab session disconnects, runs out of memory, or timeouts, re-running this notebook automatically detects the checkpoint in Drive and seamlessly continues from the exact round without losing progress.
4. **Deterministic**: Bitwise-identical RNG and state continuation verified by test suite.

## Cell 1 — Mount Google Drive

Mounts Google Drive to `/content/drive` so all training progress and checkpoints persist permanently.

In [1]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully at /content/drive")
except ImportError:
    print("Not running in Google Colab environment -- running locally.")

Mounted at /content/drive
Google Drive mounted successfully at /content/drive


## Cell 2 — Clone Repository from GitHub (No manual uploads)

Clones `Safe-Lie` from GitHub or pulls latest changes if already present, and enters the repository directory.

In [2]:
import os
import subprocess

REPO_URL = "https://github.com/aliakarma/Safe-Lie.git"
REPO_DIR = "/content/Safe-Lie"

if os.path.exists("/content") and not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} into {REPO_DIR}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.exists(REPO_DIR):
    print(f"Updating existing repo at {REPO_DIR} via git pull...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")

Cloning https://github.com/aliakarma/Safe-Lie.git into /content/Safe-Lie...
Working directory: /content/Safe-Lie


## Cell 3 — Runtime & GPU Verification

Checks CUDA availability and device specifications.

In [3]:
import os
import platform
import torch

print(f"Python version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"CPU count: {os.cpu_count()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Python version: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
CPU count: 8
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Cell 4 — Dependency Installation

Installs pinned dependencies and the editable `safelie` package.

In [4]:
!pip install -r requirements.txt
!pip install -e .

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 41.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 90.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of statsmodels to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install -r requirements.txt (line 13), -r requirements.txt (line 14), -r requirements.txt (line 8) and numpy==2.2.6 because these package versions have conflicting dependencies.

The conflict is caused b

## Cell 5 — Version Verification & Preflight Gate

Runs the fast preflight test gate to verify all modules and deterministic seeds before launching experiments.

In [8]:
import sys
import site
site.main()
sys.path.append('/content/Safe-Lie')

import numpy
import pydantic
import scipy
import torch
import safelie
from safelie.preflight import run_preflight

print(f"safelie: {safelie.__version__}")
print(f"numpy: {numpy.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"pydantic: {pydantic.VERSION}")
print(f"torch: {torch.__version__}")

preflight_exit_code = run_preflight(fast=True)
assert preflight_exit_code == 0, "Preflight gate failed -- do not proceed to training."
print("✓ Preflight passed cleanly.")

safelie: 0.1.0
numpy: 2.1.3
scipy: 1.16.3
pydantic: 2.13.4
torch: 2.11.0+cu128
[safelie.preflight] running: /usr/bin/python3 -m pytest tests/unit tests/property tests/theory tests/isolation -q (cwd=/content/Safe-Lie)
[safelie.preflight] PASS -- safe to proceed to training.
✓ Preflight passed cleanly.


## Cell 6 — Config Selection & Google Drive Output Directory

Selects the experiment configuration and routes outputs directly to Google Drive for persistence across sessions.

In [ ]:
import os
from pathlib import Path
from safelie.utils.config import load_experiment_config

# Choose experiment. pilot_A_clean .. pilot_E_clean_rce use env.name=manyagent_ant
# (real Safe-MAMuJoCo), which needs the adapter in safelie/envs/mamujoco.py -- NOT
# YET IMPLEMENTED anywhere in this repo (see that module's docstring and
# docs/reproducibility.md "Completing Stage 2"). This is a real engineering gap,
# not a local-only restriction: a Colab GPU runtime alone will not make a pilot_*
# config run -- someone must first `pip install mujoco safety-gymnasium <a
# MA-MuJoCo factorization package>` there and implement the adapter against the
# DualCostEnvWrapper contract, then re-run `pytest tests/` before spending GPU
# time. Until that adapter exists, use one of the local_demo_*/smoke configs,
# which run everywhere (CPU, GPU, Colab, laptop) on the synthetic environment and
# exercise the exact same pipeline end to end.
EXPERIMENT_NAME = "local_demo_clean"  # local_demo_clean/attack/rce/benign, smoke, or (once the adapter above is implemented) pilot_A_clean..pilot_E_clean_rce
cfg = load_experiment_config(f"configs/experiment/{EXPERIMENT_NAME}.yaml")

# Direct checkpoint storage to Google Drive if mounted
if os.path.exists("/content/drive/MyDrive"):
    DRIVE_RESULTS_DIR = "/content/drive/MyDrive/safelie_results"
    cfg = cfg.model_copy(update={"output_dir": DRIVE_RESULTS_DIR})
    print(f"Output directory routed to Google Drive: {Path(cfg.output_dir) / cfg.run_id}")
else:
    print(f"Output directory (local): {Path(cfg.output_dir) / cfg.run_id}")

print(f"Selected: {cfg.run_id} | Env: {cfg.env.name} | Steps: {cfg.total_steps} | Attack: {cfg.attack.name} | Defense: {cfg.defense.name}")

## Cell 7 — Checkpoint Inspection

Checks whether a previous checkpoint exists in Google Drive.

In [10]:
from pathlib import Path

ckpt_file = Path(cfg.output_dir) / cfg.run_id / "checkpoint.pt"
if ckpt_file.exists():
    print(f"✓ Existing checkpoint found at: {ckpt_file}")
    print("Training in Cell 8 will automatically resume from the last completed round!")
else:
    print(f"No existing checkpoint found at {ckpt_file}. Will start fresh from round 0.")

No existing checkpoint found at /content/drive/MyDrive/safelie_results/pilot_A_clean/checkpoint.pt. Will start fresh from round 0.


## Cell 8 — Run Training with Auto-Checkpoint & Auto-Resume

Executes the experiment. Saves `checkpoint.pt` every round directly to Drive. If interrupted, re-running this cell automatically picks up from the exact round.

In [13]:
from safelie.experiment import run_experiment_with_oracle

if cfg.env.name == "synthetic_constrained_marl":
    out_dir = run_experiment_with_oracle(
        cfg,
        eval_every=1,
        checkpoint_every=1,
        auto_resume=True,
    )
    print(f"Run complete/up-to-date at: {out_dir}")
else:
    try:
        out_dir = run_experiment_with_oracle(
            cfg,
            eval_every=1,
            checkpoint_every=1,
            auto_resume=True,
        )
        print(f"Run complete/up-to-date at: {out_dir}")
    except NotImplementedError as exc:
        print(f"Environment adapter needed for {cfg.env.name}: {exc}")

Environment adapter needed for manyagent_ant: env.name='manyagent_ant' requires an environment adapter not implemented in this repository. See safelie/envs/mamujoco.py for what is needed and why it was left unimplemented (a heavy, GPU-stage dependency out of scope for local CPU verification, per PROJECT_REPORT.md §R7.1).


## Cell 9 — Analysis & Summary Table

Parses the generated `rounds.jsonl` and `oracle.jsonl` files from Drive and builds the results table.

In [12]:
from pathlib import Path
from safelie.analysis.tables import build_summary_table

run_dir = Path(cfg.output_dir) / cfg.run_id
if run_dir.exists():
    print(build_summary_table({cfg.run_id: run_dir}, budget=cfg.env.budget))
else:
    print(f"No completed run found at {run_dir} yet.")

No completed run found at /content/drive/MyDrive/safelie_results/pilot_A_clean yet.
